# ทำความสะอาดข้อมูล NSL-KDD (Data Cleansing)

Notebook นี้เตรียมไฟล์ดิบ `KDDTrain+`, `KDDTest+` และ `KDDTest-21` ให้มีรูปแบบที่สม่ำเสมอและตรวจสอบได้ ก่อนส่งต่อให้ feature engineering และโมเดล ขั้นตอนถูกแยกเป็นช่วงย่อยเพื่อให้ตรวจพบข้อผิดพลาดได้ตรงจุด แทนการแก้ข้อมูลเงียบ ๆ

สิ่งที่ทำ: อ่านไฟล์ตาม schema ของ NSL-KDD, ทำชื่อหมวดหมู่/label ให้เป็นมาตรฐาน, แปลงคอลัมน์ตัวเลขและตรวจค่าผิดรูปแบบ, จัดการ missing/infinite, ลบเฉพาะแถวซ้ำทุกคอลัมน์, ตรวจช่วงค่าที่ควรเป็นไปได้, ตัด metadata/feature คงที่ และเพิ่ม target สำหรับการวิเคราะห์

ผลลัพธ์จะอยู่ใน `data-pipeline/output/NSL-KDD/clean/` พร้อมรายงานและ schema จุดสำคัญของรอบนี้: ทั้งสาม split มีข้อมูลครบและผ่านการตรวจ จึงไม่มีแถวใดถูกลบ; `difficulty` และ `num_outbound_cmds` ถูกเอาออก; และยังคงจำนวนแถวเดิมไว้ครบ การผ่าน cleansing หมายถึงข้อมูลมีรูปแบบพร้อมใช้งาน ไม่ได้หมายความว่าแต่ละ record ปลอดภัยหรือเป็นอันตราย

In [15]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## 0. กำหนด path, schema และกติกาการตรวจข้อมูล

ส่วนนี้ค้นหาโฟลเดอร์รากจากตำแหน่งที่รัน notebook แล้วกำหนดที่อยู่ของ raw data และ output จึงไม่ผูกกับ path เฉพาะเครื่อง จากนั้นระบุชื่อคอลัมน์ 43 ช่องตามลำดับในไฟล์ NSL-KDD และแยกคอลัมน์เชิงหมวดหมู่, ตัวเลข, rate และตัวเลขที่ต้องไม่ติดลบ

ตาราง/ข้อความที่ได้ยืนยันว่า notebook พบทั้ง input และ output directory ที่ถูกต้อง หาก path ไม่ถูกต้องหรือไฟล์หาย โค้ดจะหยุดพร้อมแจ้ง path ที่หาไม่พบ แทนการสร้างไฟล์ผลลัพธ์ในตำแหน่งผิด การกำหนด schema สำคัญเพราะไฟล์ดิบไม่มี header; ถ้าลำดับคอลัมน์ผิด ค่าทั้งแถวจะถูกตีความผิดชนิด

In [16]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset" / "download_nsl_dataset.py").is_file() and (candidate / "data-pipeline").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the CSI_ML_Final project root from the current working directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "dataset" / "rawdata" / "NSL-KDD"
OUTPUT_DIR = PROJECT_ROOT / "data-pipeline" / "output" / "NSL-KDD"
CLEAN_OUTPUT_DIR = OUTPUT_DIR / "clean"
CLEAN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_DIR.exists():
    raise FileNotFoundError(f"Raw data directory not found: {RAW_DIR.resolve()}")

RAW_FILES = {
    "KDDTrain": "KDDTrain+.txt",
    "KDDTest": "KDDTest+.txt",
    "KDDTest-21": "KDDTest-21.txt",
}

columns = [
    "duration",
    "protocol_type",
    "service",
    "flag",
    "src_bytes",
    "dst_bytes",
    "land",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "logged_in",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
    "num_outbound_cmds",
    "is_host_login",
    "is_guest_login",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
    "label",
    "difficulty",
]

categorical_cols = ["protocol_type", "service", "flag"]
target_col = "label"
numeric_cols = [col for col in columns if col not in categorical_cols + [target_col]]

rate_cols = [
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
]
non_negative_cols = [col for col in numeric_cols if col != "difficulty"]

print(f"Raw data directory: {RAW_DIR.relative_to(PROJECT_ROOT)}")
print(f"Cleaned output directory: {CLEAN_OUTPUT_DIR.relative_to(PROJECT_ROOT)}")

Raw data directory: dataset\rawdata\NSL-KDD
Cleaned output directory: data-pipeline\output\NSL-KDD\clean


## 1. แปลงชื่อการโจมตีเป็นกลุ่มครอบครัว

ฟังก์ชัน `add_targets` แปลงชื่อ attack รายตัวไปเป็น DoS, Probe, R2L หรือ U2R ตาม mapping ที่ระบุ; `normal` ได้ค่า family เป็น Normal; ชื่อที่ไม่รู้จักได้ Unknown เพื่อเก็บข้อมูลไว้และเปิดให้ตรวจภายหลัง นอกจากนี้ยังสร้าง `is_attack` เป็น 0 สำหรับ normal และ 1 สำหรับชื่ออื่น

การ map family ช่วยวิเคราะห์/ประเมินแบบกลุ่มได้ง่ายขึ้น แต่ไม่ควรเข้าใจว่า Unknown คือ normal: มันยังคงเป็น attack หาก label ไม่ใช่ normal และ `is_attack=1` การรักษา unknown ไว้สำคัญต่อชุดทดสอบซึ่งอาจมี attack ชนิดที่ไม่ปรากฏใน train

In [ ]:
attack_family_map = {
    "back": "DoS",
    "land": "DoS",
    "neptune": "DoS",
    "pod": "DoS",
    "smurf": "DoS",
    "teardrop": "DoS",
    "ipsweep": "Probe",
    "nmap": "Probe",
    "portsweep": "Probe",
    "satan": "Probe",
    "ftp_write": "R2L",
    "guess_passwd": "R2L",
    "imap": "R2L",
    "multihop": "R2L",
    "phf": "R2L",
    "spy": "R2L",
    "warezclient": "R2L",
    "warezmaster": "R2L",
    "buffer_overflow": "U2R",
    "loadmodule": "U2R",
    "perl": "U2R",
    "rootkit": "U2R",
    "apache2": "DoS", 
    "mailbomb": "DoS", 
    "processtable": "DoS", 
    "udpstorm": "DoS", 
    "worm": "DoS",
    "mscan": "Probe", 
    "saint": "Probe",
    "snmpguess": "R2L", 
    "snmpgetattack": "R2L",
    "httptunnel": "R2L", 
    "named": "R2L", 
    "sendmail": "R2L", 
    "xlock": "R2L", 
    "xsnoop": "R2L",
    "ps": "U2R", 
    "xterm": "U2R", 
    "sqlattack": "U2R",
    "normal": "Normal"
}

def add_targets(frame):
    frame = frame.copy()
    frame["attack_family"] = frame["label"].map(attack_family_map)
    normal_mask = frame["attack_family"].isna() & frame["label"].eq("normal")
    frame.loc[normal_mask, "attack_family"] = "Normal"
    frame["attack_family"] = frame["attack_family"].fillna("Unknown")
    frame["is_attack"] = frame["label"].ne("normal").astype("int8")
    return frame

## 2. อ่านชุดข้อมูลดิบทั้งสาม split

โค้ดอ่านไฟล์ตามชื่อที่กำหนดและนำ schema 43 คอลัมน์มาใส่ เนื่องจากข้อมูลดิบไม่มีแถวชื่อคอลัมน์ จากนั้นตารางสรุปนับจำนวนแถว คอลัมน์ ช่องว่าง และแถวซ้ำก่อนเริ่มแก้ไข

ผลที่ได้: KDDTrain 125,973 แถว, KDDTest 22,544 แถว และ KDDTest-21 11,850 แถว; ทุก split มี 43 คอลัมน์ และในไฟล์ต้นทางที่ตรวจครั้งนี้ไม่มี missing หรือ exact duplicate ขั้นนี้เป็น baseline สำหรับเทียบกับรายงานหลังทำความสะอาด

In [18]:
raw_datasets = {}
raw_summary = []

for dataset_name, filename in RAW_FILES.items():
    raw_path = RAW_DIR / filename
    if not raw_path.exists():
        raise FileNotFoundError(f"Missing raw file: {raw_path.resolve()}")

    raw_frame = pd.read_csv(raw_path, names=columns)
    raw_datasets[dataset_name] = raw_frame
    raw_summary.append({
        "dataset": dataset_name,
        "rows_raw": len(raw_frame),
        "columns_raw": raw_frame.shape[1],
        "missing_raw": int(raw_frame.isna().sum().sum()),
        "duplicates_raw": int(raw_frame.duplicated().sum()),
    })

display(pd.DataFrame(raw_summary))

,dataset,rows_raw,columns_raw,missing_raw,duplicates_raw
0,KDDTrain,125973,43,0,0
1,KDDTest,22544,43,0,0
2,KDDTest-21,11850,43,0,0


## 3. ทำมาตรฐานข้อความและชื่อหมวดหมู่

ตัดช่องว่างหัว/ท้ายและแปลง `protocol_type`, `service`, `flag` รวมถึง `label` เป็นตัวพิมพ์เล็ก ตัวอย่างเช่น `SF`, ` sf ` และ `sf` จะกลายเป็นค่าเดียวกัน ลดปัญหาที่โมเดลมองข้อความเดียวกันเป็นคนละหมวดเพราะรูปแบบตัวอักษร

ตารางรายงานจำนวน label ที่ไม่ซ้ำ: 23 ใน KDDTrain และ 38 ในแต่ละ test split จำนวนที่มากกว่าใน test บอกว่าชุดทดสอบมี attack label เพิ่มเติมจาก train ซึ่งเป็นประโยชน์ต่อการตรวจสอบการ generalize แต่ก็ทำให้ mapping family อาจมี Unknown ได้ การ normalize ไม่ได้ลบ label หรือรวม attack คนละชนิดเข้าด้วยกัน

In [19]:
normalized_datasets = {}
normalization_summary = []

for dataset_name, raw_frame in raw_datasets.items():
    frame = raw_frame.copy()

    for col in categorical_cols + [target_col]:
        frame[col] = frame[col].astype("string").str.strip().str.lower()

    normalized_datasets[dataset_name] = frame
    normalization_summary.append({
        "dataset": dataset_name,
        "normalized_columns": ", ".join(categorical_cols + [target_col]),
        "unique_labels": frame[target_col].nunique(),
    })

display(pd.DataFrame(normalization_summary))

,dataset,normalized_columns,unique_labels
0,KDDTrain,"protocol_type, service, flag, label",23
1,KDDTest,"protocol_type, service, flag, label",38
2,KDDTest-21,"protocol_type, service, flag, label",38


## 4. แปลงคอลัมน์ตัวเลขและตรวจค่าที่แปลงไม่ได้

ใช้ `to_numeric(errors=coerce)` กับคอลัมน์ที่ควรเป็นตัวเลข แล้วนับค่าที่เดิมไม่ว่างแต่แปลงไม่ได้แยกจากค่า missing ที่มีอยู่ก่อน ตรวจ `+inf/-inf` เพิ่มด้วย หากมีข้อความหรือรูปแบบตัวเลขผิด โค้ดจะหยุดทันที แทนการปล่อยให้กลายเป็น NaN โดยไม่มีใครสังเกต

ผลทั้งสามชุดเป็น 0 invalid numeric และ 0 infinite จึงไม่มีการแปลงค่าผิดรูปแบบในข้อมูลชุดนี้ ตารางนี้เป็นตัวตรวจคุณภาพของ input; ถ้าอนาคตมีค่ามากกว่า 0 ควรย้อนตรวจแถวต้นทางก่อนเลือกวิธีแทนค่าหรือลบทิ้ง

In [20]:
numeric_datasets = {}
numeric_summary = []

for dataset_name, normalized_frame in normalized_datasets.items():
    frame = normalized_frame.copy()
    invalid_numeric_values = 0

    for col in numeric_cols:
        original = frame[col]
        converted = pd.to_numeric(original, errors="coerce")
        invalid_numeric_values += int((converted.isna() & original.notna()).sum())
        frame[col] = converted

    infinite_values = int(np.isinf(frame[numeric_cols].to_numpy(dtype=float)).sum())
    numeric_datasets[dataset_name] = frame
    numeric_summary.append({
        "dataset": dataset_name,
        "invalid_numeric_values": invalid_numeric_values,
        "infinite_values": infinite_values,
    })

numeric_summary_df = pd.DataFrame(numeric_summary)
display(numeric_summary_df)

if numeric_summary_df["invalid_numeric_values"].sum() > 0:
    raise ValueError("Invalid numeric values were found; inspect numeric_summary_df before continuing.")

,dataset,invalid_numeric_values,infinite_values
0,KDDTrain,0,0
1,KDDTest,0,0
2,KDDTest-21,0,0


## 5. จัดการค่าอนันต์และแถวที่มีข้อมูลไม่ครบ

ค่าอนันต์ถูกแทนเป็น missing เพราะคำนวณ/ฝึกโมเดลกับค่าดังกล่าวได้ไม่ปลอดภัย จากนั้นนับจำนวนช่องว่างและจำนวนแถวที่มีอย่างน้อยหนึ่งช่องว่าง แล้วลบเฉพาะแถวที่ไม่สมบูรณ์ โดยเก็บจำนวนที่ลบไว้ในรายงาน

รอบนี้ทั้งสาม split มี missing ก่อนจัดการ 0 ช่อง, ลบ 0 แถว และเหลือ missing หลังจัดการ 0 ช่อง จึงไม่สูญเสียตัวอย่างจากขั้นนี้ หากข้อมูลรุ่นอื่นมี missing การลบแถวอาจทำให้ข้อมูลลดลงหรือเปลี่ยนสัดส่วนคลาส ควรตรวจรายงานก่อนยอมรับผล

In [21]:
missing_datasets = {}
missing_summary = []

for dataset_name, numeric_frame in numeric_datasets.items():
    frame = numeric_frame.replace([np.inf, -np.inf], np.nan)
    missing_cells_before = int(frame.isna().sum().sum())
    rows_dropped_missing = int(frame.isna().any(axis=1).sum())

    if rows_dropped_missing:
        frame = frame.dropna().copy()

    missing_datasets[dataset_name] = frame
    missing_summary.append({
        "dataset": dataset_name,
        "missing_cells_before": missing_cells_before,
        "rows_dropped_missing": rows_dropped_missing,
        "missing_cells_after": int(frame.isna().sum().sum()),
    })

missing_summary_df = pd.DataFrame(missing_summary)
display(missing_summary_df)

,dataset,missing_cells_before,rows_dropped_missing,missing_cells_after
0,KDDTrain,0,0,0
1,KDDTest,0,0,0
2,KDDTest-21,0,0,0


## 6. ลบแถวที่ซ้ำกันทุกคอลัมน์แบบตรงกัน

นับและลบเฉพาะแถวที่ทุกคอลัมน์เหมือนกันเป๊ะ แล้ว reset index เพื่อให้เลขแถวต่อเนื่อง รูปแบบเครือข่ายที่คล้ายกันแต่มีค่าอย่างน้อยหนึ่งช่องต่างกันจะเก็บไว้ เพราะอาจเป็นเหตุการณ์คนละรายการจริง

ผลก่อนลบเป็น 0 duplicates ในทั้งสามชุด จึงไม่มีแถวถูกตัด การ deduplicate ข้าม train/test ไม่ได้ทำในขั้นนี้ เพราะการตรวจนี้มีขอบเขตภายในแต่ละ split เท่านั้น

In [22]:
deduplicated_datasets = {}
duplicate_summary = []

for dataset_name, missing_frame in missing_datasets.items():
    duplicate_rows = int(missing_frame.duplicated().sum())
    frame = missing_frame.drop_duplicates().reset_index(drop=True)

    deduplicated_datasets[dataset_name] = frame
    duplicate_summary.append({
        "dataset": dataset_name,
        "duplicate_rows_before": duplicate_rows,
        "rows_dropped_duplicates": duplicate_rows,
        "duplicates_after": int(frame.duplicated().sum()),
    })

duplicate_summary_df = pd.DataFrame(duplicate_summary)
display(duplicate_summary_df)

,dataset,duplicate_rows_before,rows_dropped_duplicates,duplicates_after
0,KDDTrain,0,0,0
1,KDDTest,0,0,0
2,KDDTest-21,0,0,0


## 7. ตรวจช่วงค่าที่ควรเป็นไปได้

rate features ต้องอยู่ระหว่าง 0 ถึง 1 รวมปลายช่วง เพราะเป็นสัดส่วน; ตัวเลขเชิงปริมาณของ connection ต้องไม่ติดลบ ยกเว้น `difficulty` ซึ่งเป็น metadata และไม่ได้ใช้กฎเดียวกับ feature ตัวเลข โค้ดนับค่าที่ละเมิดและหยุดหากพบ เพื่อไม่ให้ค่าผิดถูกกลบด้วยการ clip โดยอัตโนมัติ

ผลตรวจพบ invalid range 0 ค่าในทุก split และ `range_details` เป็น `{}` หมายถึงไม่มีคอลัมน์ใดต้องตรวจต่อ การผ่านเกณฑ์นี้แสดงว่าค่าอยู่ในช่วงพื้นฐานที่คาดไว้ แต่ยังไม่รับประกันว่าไม่มี outlier หรือข้อมูลที่ผิดเชิงความหมาย

In [23]:
range_validated_datasets = {}
range_summary = []

for dataset_name, deduplicated_frame in deduplicated_datasets.items():
    frame = deduplicated_frame.copy()
    invalid_ranges = {}

    for col in rate_cols:
        invalid = (~frame[col].between(0, 1)) & frame[col].notna()
        if invalid.any():
            invalid_ranges[col] = int(invalid.sum())

    for col in non_negative_cols:
        invalid = (frame[col] < 0) & frame[col].notna()
        if invalid.any():
            invalid_ranges[col] = invalid_ranges.get(col, 0) + int(invalid.sum())

    invalid_range_values = sum(invalid_ranges.values())
    range_summary.append({
        "dataset": dataset_name,
        "invalid_range_values": invalid_range_values,
        "range_details": json.dumps(invalid_ranges),
    })

    if invalid_range_values:
        raise ValueError(f"{dataset_name} contains invalid ranges: {invalid_ranges}")

    range_validated_datasets[dataset_name] = frame

range_summary_df = pd.DataFrame(range_summary)
display(range_summary_df)

,dataset,invalid_range_values,range_details
0,KDDTrain,0,{}
1,KDDTest,0,{}
2,KDDTest-21,0,{}


## 8. ตัด metadata และ feature ที่ไม่มีความแปรผัน

ลบ `difficulty` เพราะเป็นระดับความยากที่ผู้จัดชุด benchmark ระบุ ไม่ใช่พฤติกรรมของ connection และลบ predictor ที่มีค่าเดียวตลอดชุด เช่น `num_outbound_cmds` ซึ่งเป็นศูนย์ทุกแถว การเก็บคอลัมน์คงที่เพิ่มมิติแต่ไม่เพิ่มสารสนเทศ และอาจทำให้ preprocessing/การวิเคราะห์สับสน

ตารางแสดงว่าแต่ละ split มีจำนวนแถวเท่าเดิม คอลัมน์ลดจาก 43 เหลือ 41 และลบสองคอลัมน์เดียวกันคือ `difficulty` กับ `num_outbound_cmds` การจัดการ schema ให้เหมือนกันทั้ง train/test ช่วยป้องกันจำนวนหรือความหมาย feature ไม่ตรงกัน

In [24]:
structurally_cleaned_datasets = {}
structure_summary = []

for dataset_name, range_frame in range_validated_datasets.items():
    frame = range_frame.copy()
    rows_before = len(frame)
    columns_before = frame.shape[1]
    dropped_columns = []

    if "difficulty" in frame.columns:
        frame = frame.drop(columns="difficulty")
        dropped_columns.append("difficulty")

    constant_columns = [
        col for col in frame.columns
        if col not in {"label"} and frame[col].nunique(dropna=False) <= 1
    ]
    if constant_columns:
        frame = frame.drop(columns=constant_columns)
        dropped_columns.extend(constant_columns)

    structurally_cleaned_datasets[dataset_name] = frame
    structure_summary.append({
        "dataset": dataset_name,
        "rows_before": rows_before,
        "rows_after": len(frame),
        "columns_before": columns_before,
        "columns_after": frame.shape[1],
        "dropped_columns": ", ".join(dropped_columns),
    })

structure_summary_df = pd.DataFrame(structure_summary)
display(structure_summary_df)

,dataset,rows_before,rows_after,columns_before,columns_after,dropped_columns
0,KDDTrain,125973,125973,43,41,"difficulty, num_outbound_cmds"
1,KDDTest,22544,22544,43,41,"difficulty, num_outbound_cmds"
2,KDDTest-21,11850,11850,43,41,"difficulty, num_outbound_cmds"


## 9. เพิ่มคอลัมน์ target สำหรับวิเคราะห์และประเมิน

`add_targets` สร้าง `attack_family` และ `is_attack` จาก `label` โดยคง label รายตัวเดิมไว้ ตารางแรกนับ normal, attack และ Unknown family; ตารางตัวอย่างแสดงรูปแบบแถวหลังเพิ่ม target

ผล: KDDTrain มี normal 67,343 และ attack 58,630; KDDTest มี normal 9,711 และ attack 12,833 โดย Unknown family 3,750; KDDTest-21 มี normal 2,152 และ attack 9,698 โดย Unknown family 3,740 ตัวเลข Unknown เป็นชนิดโจมตีที่ mapping ปัจจุบันยังไม่จัดเข้าหนึ่งใน 4 family ไม่ใช่ record ที่ไม่มี label และไม่ใช่ benign; `is_attack` ยังคงนับเป็น 1

การมี attack label ที่ไม่คุ้นใน test เป็นโจทย์สำคัญสำหรับการตรวจจับแบบ unsupervised: โมเดลไม่ควรอาศัยการจำชื่อ attack และ metric ควรรายงานผลกับกลุ่มที่ไม่เคยพบแยกจากกลุ่มอื่นด้วย

In [25]:
cleaned = {}
target_summary = []

for dataset_name, structural_frame in structurally_cleaned_datasets.items():
    frame = add_targets(structural_frame)
    cleaned[dataset_name] = frame

    target_summary.append({
        "dataset": dataset_name,
        "rows": len(frame),
        "normal_rows": int((frame["is_attack"] == 0).sum()),
        "attack_rows": int((frame["is_attack"] == 1).sum()),
        "unknown_attack_labels": int((frame["attack_family"] == "Unknown").sum()),
    })

display(pd.DataFrame(target_summary))
display(cleaned["KDDTrain"].head())

,dataset,rows,normal_rows,attack_rows,unknown_attack_labels
0,KDDTrain,125973,67343,58630,0
1,KDDTest,22544,9711,12833,3750
2,KDDTest-21,11850,2152,9698,3740


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,attack_family,is_attack
0,0,tcp,ftp_data,sf,491,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,0.000,0.000,0.000,0.000,1.000,0.000,0.000,150,25,0.170,0.030,0.170,0.000,0.000,0.000,0.050,0.000,normal,Normal,0
1,0,udp,other,sf,146,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,1,0.000,0.000,0.000,0.000,0.080,0.150,0.000,255,1,0.000,0.600,0.880,0.000,0.000,0.000,0.000,0.000,normal,Normal,0
2,0,tcp,private,s0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,123,6,1.000,1.000,0.000,0.000,0.050,0.070,0.000,255,26,0.100,0.050,0.000,0.000,1.000,1.000,0.000,0.000,neptune,DoS,1
3,0,tcp,http,sf,232,8153,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,5,5,0.200,0.200,0.000,0.000,1.000,0.000,0.000,30,255,1.000,0.000,0.030,0.040,0.030,0.010,0.000,0.010,normal,Normal,0
4,0,tcp,http,sf,199,420,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,30,32,0.000,0.000,0.000,0.000,1.000,0.000,0.090,255,255,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,normal,Normal,0


## 10. ตรวจไฟล์หลัง cleansing ก่อนบันทึก

ตาราง final summary ตรวจจำนวนแถว/คอลัมน์ missing, duplicate และยืนยันว่า `difficulty` กับ `num_outbound_cmds` ไม่หลงเหลือ หากเงื่อนไขใดไม่ผ่าน โค้ดจะหยุดก่อน export เพื่อไม่สร้างไฟล์ที่ดูเหมือนสำเร็จแต่มีปัญหา

ผลปัจจุบัน: ทั้งสามชุดคงจำนวนแถวเดิม, ไม่มี missing/duplicate และไม่มีสองคอลัมน์ที่ต้องตัด ค่า `columns_final=43` ไม่ขัดกับตารางก่อนหน้าที่มี 41 คอลัมน์ เพราะหลังตัดเหลือ 41 แล้วขั้น target เพิ่ม `attack_family` กับ `is_attack` อีก 2 คอลัมน์ รวมกลับเป็น 43

In [26]:
final_summary = []

for dataset_name, frame in cleaned.items():
    final_summary.append({
        "dataset": dataset_name,
        "rows_final": len(frame),
        "columns_final": frame.shape[1],
        "missing_final": int(frame.isna().sum().sum()),
        "duplicates_final": int(frame.duplicated().sum()),
        "has_difficulty": "difficulty" in frame.columns,
        "has_constant_num_outbound_cmds": "num_outbound_cmds" in frame.columns,
    })

final_summary_df = pd.DataFrame(final_summary)
display(final_summary_df)

if (
    final_summary_df["missing_final"].sum() > 0
    or final_summary_df["duplicates_final"].sum() > 0
    or final_summary_df["has_difficulty"].any()
    or final_summary_df["has_constant_num_outbound_cmds"].any()
):
    raise ValueError("Final validation failed; inspect final_summary_df.")

,dataset,rows_final,columns_final,missing_final,duplicates_final,has_difficulty,has_constant_num_outbound_cmds
0,KDDTrain,125973,43,0,0,False,False
1,KDDTest,22544,43,0,0,False,False
2,KDDTest-21,11850,43,0,0,False,False


## 11. บันทึกข้อมูลสะอาด รายงาน และ metadata

บันทึก CSV ที่ผ่านการตรวจแล้วแยกตาม split, รวมสถิติจากแต่ละขั้นเป็น `cleaning_report.csv`, บันทึกชื่อคอลัมน์และ dtype ใน `cleaned_schema.csv` และบันทึกกติกาการทำงานไว้ใน `cleaning_config.json`

รายงานที่แสดงด้านล่างใช้ตรวจย้อนกลับได้ว่าแต่ละ split มีการลบแถว/คอลัมน์เท่าไรและผ่าน validation หรือไม่ รอบนี้จำนวนแถวถูกลบทุกขั้นเป็นศูนย์ ส่วนคอลัมน์ที่ลบมีสองรายการที่ระบุไว้ รายการ output ถูกเก็บในโฟลเดอร์ `clean/` และถูกละเว้นจาก Git เพราะสร้างใหม่จาก raw data ได้

In [27]:
report_df = (
    pd.DataFrame(raw_summary)
    .merge(numeric_summary_df, on="dataset")
    .merge(missing_summary_df, on="dataset")
    .merge(duplicate_summary_df, on="dataset")
    .merge(range_summary_df, on="dataset")
    .merge(structure_summary_df, on="dataset")
    .merge(final_summary_df, on="dataset")
)

for dataset_name, frame in cleaned.items():
    output_path = CLEAN_OUTPUT_DIR / f"{dataset_name}_clean.csv"
    frame.to_csv(output_path, index=False)
    print(f"Saved {output_path.relative_to(PROJECT_ROOT)}")

report_df.to_csv(CLEAN_OUTPUT_DIR / "cleaning_report.csv", index=False)

schema_df = pd.DataFrame({
    "column": cleaned["KDDTrain"].columns,
    "dtype": [str(dtype) for dtype in cleaned["KDDTrain"].dtypes],
})
schema_df.to_csv(CLEAN_OUTPUT_DIR / "cleaned_schema.csv", index=False)

config = {
    "raw_directory": str(RAW_DIR.resolve()),
    "output_directory": str(CLEAN_OUTPUT_DIR.resolve()),
    "steps": [
        "normalize_text",
        "convert_numeric",
        "replace_infinite_and_drop_missing_rows",
        "drop_exact_duplicates",
        "validate_ranges",
        "drop_difficulty_and_constant_columns",
        "add_attack_family_and_is_attack",
    ],
    "metadata_columns_removed": ["difficulty"],
    "constant_columns_removed": ["num_outbound_cmds"],
    "categorical_columns_preserved": categorical_cols,
    "target_columns_added": ["attack_family", "is_attack"],
}
with (CLEAN_OUTPUT_DIR / "cleaning_config.json").open("w", encoding="utf-8") as file:
    json.dump(config, file, indent=2)

display(report_df)

Saved data-pipeline\output\NSL-KDD\clean\KDDTrain_clean.csv
Saved data-pipeline\output\NSL-KDD\clean\KDDTest_clean.csv
Saved data-pipeline\output\NSL-KDD\clean\KDDTest-21_clean.csv


,dataset,rows_raw,columns_raw,missing_raw,duplicates_raw,invalid_numeric_values,infinite_values,missing_cells_before,rows_dropped_missing,missing_cells_after,duplicate_rows_before,rows_dropped_duplicates,duplicates_after,invalid_range_values,range_details,rows_before,rows_after,columns_before,columns_after,dropped_columns,rows_final,columns_final,missing_final,duplicates_final,has_difficulty,has_constant_num_outbound_cmds
0,KDDTrain,125973,43,0,0,0,0,0,0,0,0,0,0,0,{},125973,125973,43,41,"difficulty, num_outbound_cmds",125973,43,0,0,False,False
1,KDDTest,22544,43,0,0,0,0,0,0,0,0,0,0,0,{},22544,22544,43,41,"difficulty, num_outbound_cmds",22544,43,0,0,False,False
2,KDDTest-21,11850,43,0,0,0,0,0,0,0,0,0,0,0,{},11850,11850,43,41,"difficulty, num_outbound_cmds",11850,43,0,0,False,False


## 12. สรุปไฟล์ที่สร้างและการกระจาย target หลัง cleansing

ส่วนแรกแสดงชื่อ/ขนาดไฟล์ที่ถูกสร้างในโฟลเดอร์ `clean/`; ส่วนท้ายแสดงจำนวนและเปอร์เซ็นต์ของแต่ละ attack family ใน KDDTrain เพื่อให้ตรวจว่าการเพิ่ม target ทำงานและสัดส่วนยังสอดคล้องกับ EDA

ผล family ควรตรงกับที่สำรวจไว้ก่อนหน้า: Normal 53.46%, DoS 36.46%, Probe 9.25%, R2L 0.79% และ U2R 0.04% หากค่าไม่ตรงอย่างมีนัยสำคัญให้ตรวจ label normalization, mapping และจำนวนแถวก่อนส่งข้อมูลต่อ ขั้นนี้ไม่ได้ทำ balancing หรือ oversampling; จึงรักษาการกระจายเดิมไว้

In [28]:
print("Generated files:")
for path in sorted(CLEAN_OUTPUT_DIR.iterdir()):
    print(f"- {path.name}: {path.stat().st_size:,} bytes")

print("\nCleaned training attack-family distribution:")
display(
    cleaned["KDDTrain"]["attack_family"]
    .value_counts()
    .rename_axis("attack_family")
    .reset_index(name="rows")
    .assign(percent=lambda x: x["rows"] / len(cleaned["KDDTrain"]) * 100)
)

Generated files:
- cleaned_schema.csv: 963 bytes
- cleaning_config.json: 799 bytes
- cleaning_report.csv: 788 bytes
- KDDTest-21_clean.csv: 1,725,717 bytes
- KDDTest_clean.csv: 3,264,293 bytes
- KDDTrain_clean.csv: 18,073,730 bytes

Cleaned training attack-family distribution:


,attack_family,rows,percent
0,Normal,67343,53.458
1,DoS,45927,36.458
2,Probe,11656,9.253
3,R2L,995,0.790
4,U2R,52,0.041
